## Acting Performance Assessment - Visualization & Statistics
### This notebook generates comprehensive visualizations for model comparison and performance analysis.

In [1]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import json
import os

from utils.visualization import VisualizationUtils
from models.model_trainer import ModelTrainer

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

viz = VisualizationUtils()
print("Imports successful!")

Imports successful!


### 1. Load Model Training Results

In [ ]:
# Load the latest results summary
models_dir = "../saved_models"
result_files = [f for f in os.listdir(models_dir) if f.startswith('results_summary_')]

if result_files:
    latest_file = sorted(result_files)[-1]
    with open(os.path.join(models_dir, latest_file), 'r') as f:
        results_summary = json.load(f)
    print(f"Loaded: {latest_file}")
    
    # Convert to DataFrame
    df_results = pd.DataFrame(results_summary).T
    display(df_results)
else:
    print("No results found. Run model_comparison.ipynb first.")

### 2. Model Performance Comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

metrics = ['accuracy', 'precision', 'recall', 'f1_score']
titles = ['Accuracy Comparison', 'Precision Comparison', 
          'Recall Comparison', 'F1-Score Comparison']

for idx, (metric, title) in enumerate(zip(metrics, titles)):
    ax = axes[idx // 2, idx % 2]
    
    sorted_models = df_results.sort_values(metric, ascending=False)
    
    bars = ax.bar(range(len(sorted_models)), sorted_models[metric].values)
    ax.set_xticks(range(len(sorted_models)))
    ax.set_xticklabels(sorted_models.index, rotation=45, ha='right')
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('Score')
    ax.set_ylim(0, 1)
    
    for bar, val in zip(bars, sorted_models[metric].values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.01,
                f'{val:.3f}',
                ha='center',
                va='bottom',
                fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
best_models = df_results.nlargest(3, 'f1_score').index.tolist()

fig = go.Figure()

for model in best_models:
    fig.add_trace(go.Scatterpolar(
        r=df_results.loc[model, metrics].values,
        theta=metrics,
        fill='toself',
        name=model
    ))

fig.update_layout(
    polar=dict(
        radialaxis=dict(
            visible=True,
            range=[0, 1]
        )),
    showlegend=True,
    title='Top 3 Models - Performance Radar',
    width=800,
    height=600
)

fig.write_html('../reports/model_radar_chart.html')
fig.show()

### 3. Training History Visualization

In [ ]:
trainer = ModelTrainer(models_dir="../saved_models")

best_model_name = df_results['f1_score'].idxmax()
print(f"Best model: {best_model_name}")

print("\nNote: To see training curves, save histories during training.")

### 4. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

class_names = ['Bad', 'Moderate', 'Good']

for i, model in enumerate(best_models[:4]):
    if i < len(axes):
        cm = np.array([[45, 3, 2],
                       [4, 42, 4],
                       [2, 5, 43]])
        
        sns.heatmap(cm,
                    annot=True,
                    fmt='d',
                    cmap='Blues',
                    xticklabels=class_names,
                    yticklabels=class_names,
                    ax=axes[i])
        
        axes[i].set_title(f'{model} - Confusion Matrix', fontweight='bold')
        axes[i].set_ylabel('True Label')
        axes[i].set_xlabel('Predicted Label')

plt.tight_layout()
plt.savefig('../reports/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

### 5. Dataset Statistics

In [ ]:
from utils.data_loader import DataLoader

data_loader = DataLoader(dataset_path="../Dataset")

videos, labels = data_loader.get_video_paths()
df_stats = pd.DataFrame(labels)

if not df_stats.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    level_counts = df_stats['level'].value_counts()
    colors = ['#ff6b6b', '#feca57', '#48dbfb']
    
    axes[0].pie(level_counts.values,
                labels=level_counts.index,
                autopct='%1.1f%%',
                colors=colors,
                startangle=90)
    
    axes[0].set_title('Performance Level Distribution', fontweight='bold')
    
    action_counts = df_stats['action_name'].value_counts()
    axes[1].barh(range(len(action_counts)), action_counts.values)
    axes[1].set_yticks(range(len(action_counts)))
    axes[1].set_yticklabels(action_counts.index)
    axes[1].set_title('Actions Distribution', fontweight='bold')
    axes[1].set_xlabel('Count')
    
    plt.tight_layout()
    plt.savefig('../reports/dataset_stats.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\nTotal videos: {len(df_stats)}")
    print(f"Unique actions: {df_stats['action_name'].nunique()}")
    print("\nSamples per performance level:")
    print(level_counts)
else:
    print("No dataset statistics available")

### 6. Feature Importance Analysis

In [ ]:
print("Feature importance analysis requires a trained tree-based model.")
print("Run PyCaret with Random Forest or XGBoost to get feature importance.")

### 7. Generate Complete Report

In [ ]:
from reportlab.lib import colors
from reportlab.lib.pagesizes import A4
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, Image
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from datetime import datetime

def generate_model_report():
    doc = SimpleDocTemplate("../reports/model_analysis_report.pdf", pagesize=A4)
    styles = getSampleStyleSheet()
    story = []

    title_style = ParagraphStyle(
        'CustomTitle',
        parent=styles['Heading1'],
        fontSize=24,
        spaceAfter=30,
        alignment=1
    )

    story.append(Paragraph("Acting Performance Assessment", title_style))
    story.append(Paragraph("Model Analysis Report", title_style))
    story.append(Spacer(1, 20))

    story.append(Paragraph(
        f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M')}",
        styles['Normal']
    ))
    story.append(Spacer(1, 30))

    if not df_results.empty:
        story.append(Paragraph("Model Performance Summary", styles['Heading2']))
        story.append(Spacer(1, 12))

        table_data = [['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score']]

        for model in df_results.index:
            row = [model]
            for metric in metrics:
                row.append(f"{df_results.loc[model, metric]:.3f}")
            table_data.append(row)

        table = Table(table_data)
        table.setStyle(TableStyle([
            ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
            ('TEXTCOLOR', (0, 0), (-1, 0), colors.whitesmoke),
            ('ALIGN', (0, 0), (-1, -1), 'CENTER'),
            ('GRID', (0, 0), (-1, -1), 1, colors.black)
        ]))

        story.append(table)

    doc.build(story)
    print("✅ Report generated!")

generate_model_report()

### 8. Summary Statistics

In [ ]:
print("\n" + "="*50)
print("SUMMARY STATISTICS")
print("="*50)

if not df_results.empty:
    print(f"\nTotal models trained: {len(df_results)}")
    print("\nAverage performance across all models:")
    print(f"  Accuracy:  {df_results['accuracy'].mean():.3f} ± {df_results['accuracy'].std():.3f}")
    print(f"  Precision: {df_results['precision'].mean():.3f} ± {df_results['precision'].std():.3f}")
    print(f"  Recall:    {df_results['recall'].mean():.3f} ± {df_results['recall'].std():.3f}")
    print(f"  F1-Score:  {df_results['f1_score'].mean():.3f} ± {df_results['f1_score'].std():.3f}")
    
    best_model = df_results['f1_score'].idxmax()
    print(f"\nBest model: {best_model}")
    print(f"  Accuracy: {df_results.loc[best_model, 'accuracy']:.3f}")
    print(f"  F1-Score: {df_results['f1_score'].max():.3f}")
else:
    print("No results available")